# RNN CafeBERT 512, category: cw effective β=0.999 + label smoothing 0.05

Cùng cấu hình với `dl-cat-cafe-rnn-ce-s42..44` (Bi-GRU ‖ Bi-LSTM → CNN, h=100, c=50, lr 3e-4, wd 0.01,
patience 8 theo dev macro-F1, `best.pt` ở đỉnh). Khác duy nhất là loss:
`--class-weight --cw-mode effective --cw-beta 0.999 --label-smoothing 0.05`.

Mốc so (dev, CE): s42 0,6172 · s43 0,6223 · s44 0,6060.

**Trước khi chạy:** Runtime → Change runtime type → **GPU L4 hoặc A100** (T4 nhiều khả năng quá chậm).
Cần trên Drive của tài khoản này: `MyDrive/vietjobs_splits/` (train.csv dev.csv test.csv manifest.json).
Cache token ≈ 40 GB nằm trên đĩa Colab, không lưu lên Drive; mỗi phiên nhúng lại.

In [ ]:
# Chia seed giữa hai tài khoản, ví dụ tài khoản A: [42, 43], tài khoản B: [44]
SEEDS = [42, 43]

In [ ]:
!nvidia-smi | head -15
!df -h /content | tail -1
!free -g | head -2

In [ ]:
import os
if not os.path.isdir('vietjobs'):
    !git clone -q -b t8-rnn-colab https://github.com/boy2407/vietjobs.git
%cd vietjobs
!git log --oneline -1
# KHÔNG cài lại torch — giữ bản CUDA sẵn có của Colab.
!pip install -q "transformers>=4.38,<4.47" "pyarrow>=15.0" "scikit-learn>=1.5" \
    "scipy>=1.13" "joblib>=1.4" "underthesea>=6.8" sentencepiece protobuf pytest
!pip install -q -e . --no-deps
%env PYTHONPATH=src

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert '--cw-mode' in open('src/vietjobs/dl/train_dl.py').read(), 'train_dl.py chưa có --cw-mode'
assert '--label-smoothing' in open('src/vietjobs/dl/train_dl.py').read(), 'train_dl.py chưa có --label-smoothing'
print('train_dl.py có --cw-mode và --label-smoothing')

## Khôi phục splits và chạy test

In [ ]:
!python notebooks/colab_setup.py --family raw --no-train-token --skip-tests

In [ ]:
!python -m pytest -q tests/test_dl_encode.py tests/test_dl_heads.py

## Nhúng cache token CafeBERT 512 (category, train + dev)

`--pooling none` ghi `[n, 512, 1024]` float16 qua memmap. `dev` trước để có bằng chứng sớm. Hết VRAM thì giảm `--batch`.

In [ ]:
!python -m vietjobs.dl.encode --task category --splits dev --encoder cafebert --max-len 512 --pooling none --device cuda --batch 32

In [ ]:
!python -m vietjobs.dl.encode --task category --splits train --encoder cafebert --max-len 512 --pooling none --device cuda --batch 32

In [ ]:
import glob, numpy as np
for split, n in (('train', 34354), ('dev', 3812)):
    hits = glob.glob(f'artifacts/embeddings/**/cafebert-{split}-raw-len512-tok.f16.npy', recursive=True)
    assert hits, f'chưa có cache token {split}'
    a = np.load(hits[0], mmap_mode='r')
    assert a.shape == (n, 512, 1024), (hits[0], a.shape)
    print(hits[0], a.shape, a.dtype)
!df -h /content | tail -1

## Huấn luyện

Mỗi seed một lệnh. Theo dõi `seconds` của epoch đầu: nếu một epoch quá lâu so với thời gian còn lại của phiên, dừng và đổi GPU.
Sau mỗi seed, chạy ngay cell lưu kết quả bên dưới.

In [ ]:
import subprocess, json
def train(seed):
    rid = f'dl-cat-cafe-rnn-cweff999-ls05-s{seed}'
    cmd = (f'python -m vietjobs.dl.train_dl --task category --head rnn --encoder cafebert --max-len 512 '
           f'--device cuda --hidden 100 --conv-channels 50 --seed {seed} '
           f'--class-weight --cw-mode effective --cw-beta 0.999 --label-smoothing 0.05 '
           f'--run-id {rid} 2>&1 | tee logs/{rid}.txt')
    os.makedirs('logs', exist_ok=True)
    subprocess.run(cmd, shell=True, check=True)
    m = json.load(open(f'artifacts/{rid}/metrics.json'))
    print(rid, 'best', m['best_epoch'], '/', m['epochs_run'],
          {k: round(v, 4) for k, v in m['metrics'].items() if isinstance(v, float)})
    return rid

In [ ]:
import shutil
from google.colab import files
def save(rid):
    out = f'/content/zip_{rid}'
    os.makedirs(out, exist_ok=True)
    shutil.copytree(f'artifacts/{rid}', f'{out}/artifacts/{rid}', dirs_exist_ok=True)
    os.makedirs(f'{out}/logs', exist_ok=True)
    shutil.copy(f'logs/{rid}.txt', f'{out}/logs/')
    # chỉ các dòng log của run này, để chép vào docs/04-results.md trên Mac (append-only)
    rows = [l for l in open('docs/04-results.md', encoding='utf-8') if f'| {rid} |' in l]
    open(f'{out}/04-results-rows.md', 'w', encoding='utf-8').writelines(rows)
    zp = shutil.make_archive(f'/content/{rid}', 'zip', out)
    os.makedirs('/content/drive/MyDrive/vietjobs_runs', exist_ok=True)
    shutil.copy(zp, '/content/drive/MyDrive/vietjobs_runs/')
    print('đã lưu', zp, '→ MyDrive/vietjobs_runs/')
    files.download(zp)

In [ ]:
for s in SEEDS:
    rid = train(s)
    save(rid)

## Mang về máy chính

Giải nén từng `dl-cat-cafe-rnn-cweff999-ls05-s<seed>.zip` vào gốc repo trên Mac (tạo `artifacts/<run_id>/` và `logs/<run_id>.txt`);
`04-results-rows.md` là dòng log để chép vào `docs/04-results.md`. Báo lại để so cặp với `dl-cat-cafe-rnn-ce-s42..44`.